# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}\n\nDescription: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

All components (record sets, fields, columns) are referenced using their unique `@id` as per Croissant schema.

In [ ]:
# Print available record sets and their fields
record_sets_meta = getattr(dataset.metadata, 'recordSet', [])
if not record_sets_meta:
    print("No record sets found in the metadata. The dataset may require reading file objects or distributions directly.")

else:
    for rs in record_sets_meta:
        print("RecordSet @id:", rs['@id'])
        print("  Name:", rs.get('name', 'N/A'))
        fields = rs.get('field', [])
        if fields:
            for f in fields:
                print("    Field @id:", f['@id'], "| Name:", f.get('name', 'N/A'))
                columns = f.get('column', [])
                if columns:
                    for c in columns:
                        print("      Column @id:", c['@id'], "| Name:", c.get('name', 'N/A'))
        print("*")

# If no record sets are found, list available distributions/files
if not record_sets_meta:
    distributions = getattr(dataset.metadata, 'distribution', [])
    print("Available distributions:")
    for d in distributions:
        print("Distribution @id:", d['@id'])

## 3. Data Extraction
Load data from each record set into DataFrames for analysis.
Use the record set and field `@id`s from the overview. All accesses are variable-driven and reference `@id`s.

In [ ]:
dataframes = {}
record_sets_meta = getattr(dataset.metadata, 'recordSet', [])
record_set_ids = []

# If record sets are defined in metadata, extract them
for rs in record_sets_meta:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id}: Columns - {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Failed to extract records for {record_set_id}: {e}")

# If no record sets are present, attempt to extract from distributions
if not record_set_ids:
    distributions = getattr(dataset.metadata, 'distribution', [])
    for d in distributions:
        dist_id = d['@id']
        try:
            records = list(dataset.records(distribution=dist_id))
            df = pd.DataFrame(records)
            dataframes[dist_id] = df
            print(f"Loaded DataFrame for distribution {dist_id}: Columns - {df.columns.tolist()}")
            print(df.head())
        except Exception as e:
            print(f"Failed to extract records for distribution {dist_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This part references fields and columns by their `@id` and uses variables for processing.

In [ ]:
# Choose a record set and numeric field to analyze
selected_record_set = record_set_ids[0] if record_set_ids else list(dataframes.keys())[0]
df = dataframes[selected_record_set]

# Try to pick a numeric field by name or logic
numeric_field_name_candidates = [col for col in df.columns if ("score" in col.lower() or "age" in col.lower() or col.lower().endswith("_7") or col.lower().endswith("_9"))]
if numeric_field_name_candidates:
    numeric_field_id = numeric_field_name_candidates[0]
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0]

threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a group field
group_field_candidates = [col for col in df.columns if ("gender" in col.lower() or "village" in col.lower() or "education" in col.lower() or "marital" in col.lower())]
if group_field_candidates:
    group_field = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
All field references use their `@id` or column names extracted earlier.

In [ ]:
# Distribution plot for the selected numeric field
plt.figure(figsize=(8, 5))
df[numeric_field_id].hist(bins=20)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Grouped bar plot
if group_field_candidates:
    grouped = df.groupby(group_field)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(8, 5))
    plt.bar(grouped[group_field], grouped[numeric_field_id])
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset demonstrates the application of AI-ready data standards using Croissant.
- Data on mental health indicators (e.g., GAD-7, PHQ-9 scores) is available at the individual record level, including key demographics.
- Simple EDA reveals potential for deeper analysis by demographic grouping and normalization.
- Visualizations uncover data distribution trends, supporting community health initiatives and further research.